# Artificial Intelligence — Lab 5
## Hill Climbing and Random-Restart Local Search

**Course Learning Outcome — CLO3**  
Illustrate and analyze the performance of local, global, and optimal search methods.

**Environment:** Python 3 / Jupyter Notebook  
**Submission:** completed notebook containing predictions, traces, code, experiments, justifications, debugging answers, and reflection.

> **Assessment principle:** Correct code is only one part of the evidence. Most marks come from your ability to **predict local-search behavior, justify neighborhood and objective choices, interpret failures, compare multiple runs, and explain why random restart can improve robustness**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Local-search concepts | 15 min | Identify objective, state, neighborhood, and local optima |
| 2. Manual hill-climbing trace | 20 min | Predict moves before implementation |
| 3. Implement hill climbing | 30 min | Build steepest-ascent local search |
| 4. Diagnose local maxima and plateaus | 20 min | Explain failure modes |
| 5. Random-restart experiment | 25 min | Measure success rate across many starts |
| 6. Debugging, variation & reflection | 10 min | Defend conclusions and diagnose faulty logic |

> **Main idea:** Hill climbing uses only local information. It can be efficient, but its result depends strongly on the starting state and the structure of the search landscape.

## Learning Objectives

By the end of this lab, you should be able to:

1. distinguish a **state-space search path problem** from a **local optimization problem**;
2. define an objective function and a neighborhood relation;
3. manually trace steepest-ascent hill climbing;
4. implement hill climbing in Python;
5. identify local maxima, plateaus, and ridges;
6. explain why hill climbing is not generally globally optimal;
7. implement random-restart hill climbing;
8. measure empirical success rate and solution quality over many runs;
9. diagnose common local-search implementation errors;
10. justify when random restart is useful.

In [ ]:
import math
import random
import statistics
import matplotlib.pyplot as plt

print("Lab 5 environment ready.")

# Part I — Local Search as Optimization

In systematic search, we often care about a **path** from an initial state to a goal.

In local search, we often care primarily about the **quality of the current state**.

For a maximization problem, let

$$
f(s)
$$

be the objective value of state $s$.

A local-search algorithm repeatedly moves to a neighboring state that appears better.

For this lab, the state is an integer

$$
x \in \{0,1,\ldots,100\}
$$

and the legal neighbors are usually

$$
N(x)=\{x-1,x+1\}
$$

when those values remain inside the domain.

## Task 1.1 — State, Objective, Neighborhood

Answer in your own words.

1. What is the **state representation** in this problem?
2. What does the objective function measure?
3. What does it mean for one state to be a **neighbor** of another?
4. Why does hill climbing usually not store a full search tree?
5. In a local-search optimization problem, why might the final state matter more than the path used to reach it?

**Your answers:**

# Part II — A Rugged Search Landscape

We will optimize the following function:

$$
f(x)=
8\sin\left(\frac{x}{6}\right)
+
5\sin\left(\frac{x}{2.7}\right)
+
0.08x
$$

for integer values $x \in [0,100]$.

The function contains several peaks, so a local search may stop at different points depending on where it starts.

In [ ]:
def objective(x: int) -> float:
    return (
        8 * math.sin(x / 6)
        + 5 * math.sin(x / 2.7)
        + 0.08 * x
    )

DOMAIN_MIN = 0
DOMAIN_MAX = 100

xs = list(range(DOMAIN_MIN, DOMAIN_MAX + 1))
ys = [objective(x) for x in xs]

plt.figure(figsize=(10, 4))
plt.plot(xs, ys)
plt.xlabel("State x")
plt.ylabel("Objective f(x)")
plt.title("Local-Search Landscape")
plt.grid(True, alpha=0.25)
plt.show()

## Task 2.1 — Inspect the Landscape

Without running hill climbing yet, inspect the plot and answer:

1. Approximately how many visible peaks can you identify?
2. Which peak appears to be the **global maximum**?
3. Give one approximate interval that contains a **local maximum that is not global**.
4. Why is this landscape potentially difficult for hill climbing?

**Your answers:**

In [ ]:
def neighbors(x: int):
    result = []
    if x > DOMAIN_MIN:
        result.append(x - 1)
    if x < DOMAIN_MAX:
        result.append(x + 1)
    return result

# Part III — Manual Hill-Climbing Trace

We use **steepest-ascent hill climbing**.

At each step:

1. inspect all neighbors;
2. choose the neighbor with the highest objective value;
3. move only if that neighbor is strictly better than the current state;
4. stop if no improving neighbor exists.

Formally, if

$$
x'=\arg\max_{y\in N(x)} f(y),
$$

move to $x'$ only when

$$
f(x') > f(x).
$$

## Task 3.1 — Predict a Trace Before Coding

Use the starting state:

```text
x = 10
```

For the first four iterations, compute the objective values of the current state and its neighbors.

| Iteration | Current $x$ | $f(x)$ | Left neighbor value | Right neighbor value | Predicted next state |
|---:|---:|---:|---:|---:|---:|
| 0 | 10 |  |  |  |  |
| 1 |  |  |  |  |  |
| 2 |  |  |  |  |  |
| 3 |  |  |  |  |  |

Then answer:

1. In which direction do you expect the algorithm to move initially?
2. Do you expect it to reach the global maximum from this start? Why or why not?

**Your prediction:**

# Part IV — Implement Steepest-Ascent Hill Climbing

Your function should return:

```python
(final_state, final_value, trace)
```

where `trace` records every visited state and its objective value.

In [ ]:
def hill_climb(start: int):
    current = start
    trace = [(current, objective(current))]

    while True:
        # TODO 1: generate the neighbors of current

        # TODO 2: select the neighbor with the highest objective value

        # TODO 3:
        # if the best neighbor is not strictly better than current,
        # stop and return the result

        # TODO 4:
        # otherwise move to the best neighbor and append it to trace
        pass

## Task 4.1 — Predict Before Running

For `start = 10`, write:

- **Predicted final state:**  
- **Predicted reason for stopping:**  
- **Do you expect global optimality?**  

Then run the next cell.

In [ ]:
start = 10
final_state, final_value, trace = hill_climb(start)

print("Start state:", start)
print("Final state:", final_state)
print("Final objective:", round(final_value, 4))
print("Number of moves:", len(trace) - 1)
print("Trace:", trace)

## Task 4.2 — Explain the Trace

Answer:

1. Did your manual prediction match the program?
2. Why did the algorithm stop where it did?
3. Is the stopping state a **local maximum**?
4. How could you verify whether it is the **global maximum**?
5. Why does hill climbing not automatically know about a better peak elsewhere?

**Your answers:**

In [ ]:
best_global_x = max(xs, key=objective)
best_global_value = objective(best_global_x)

print("Global optimum by exhaustive inspection:")
print("x* =", best_global_x)
print("f(x*) =", round(best_global_value, 4))

## Task 4.3 — Local vs. Global Optimum

Compare your hill-climbing result with the exhaustive global optimum.

| Quantity | Hill climbing from 10 | Global optimum |
|---|---:|---:|
| State |  |  |
| Objective value |  |  |

Then explain:

> Why is the hill-climbing solution valid as a **local optimum** even when it is not globally optimal?

**Your answer:**

# Part V — Starting State Sensitivity

Hill climbing can reach different peaks from different starting states.

## Task 5.1 — Predict Multiple Starts

Before running them, predict whether each start is likely to reach the same or different final peaks:

```text
5, 20, 35, 55, 75, 95
```

Write a short prediction for at least three of the starts.

**Your prediction:**

In [ ]:
starts = [5, 20, 35, 55, 75, 95]

results = []

for s in starts:
    end_x, end_value, trace = hill_climb(s)
    results.append((s, end_x, end_value, len(trace) - 1))

print("start | final | value | moves")
for row in results:
    print(
        f"{row[0]:>5} | {row[1]:>5} | "
        f"{row[2]:>7.3f} | {row[3]:>5}"
    )

## Task 5.2 — Interpret Starting-State Sensitivity

Complete:

| Start | Final state | Final objective | Global optimum reached? |
|---:|---:|---:|---|
| 5 |  |  |  |
| 20 |  |  |  |
| 35 |  |  |  |
| 55 |  |  |  |
| 75 |  |  |  |
| 95 |  |  |  |

Then answer:

1. Did all starts reach the same peak?
2. Which starts reached the global optimum?
3. What does this demonstrate about hill climbing?
4. Is the algorithm deterministic once the start state is fixed in this implementation?

**Your answers:**

# Part VI — Local Maxima, Plateaus, and Ridges

Three important failure modes are:

### Local maximum
A state is better than all immediate neighbors but not globally best.

### Plateau
Many neighboring states have the same objective value, so there is no obvious uphill direction.

### Ridge
Improvement may require a sequence of moves that is not well aligned with the available neighborhood directions.

## Task 6.1 — Conceptual Diagnosis

For each situation below, identify the main difficulty.

1. Every immediate neighbor has a lower value, but a much higher peak exists far away.
2. Several adjacent states have exactly the same value.
3. A better region exists diagonally, but the allowed moves are only horizontal and vertical.

| Situation | Local maximum / Plateau / Ridge | Justification |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |

**Your answers:**

## Task 6.2 — Plateau Experiment

Consider:

```python
plateau_value(x) = 10
```

for every $x$ from 40 through 45, while the values immediately outside that region are lower.

With our rule requiring **strict improvement**, answer:

1. What happens if hill climbing enters the plateau?
2. Why?
3. Would allowing sideways moves change the behavior?
4. What new risk can unlimited sideways moves create?

**Your answer:**

# Part VII — Random-Restart Hill Climbing

A common remedy for local maxima is to run hill climbing multiple times from different random initial states.

The best result across all runs is retained.

If one run succeeds with probability $p$, then after $k$ independent restarts the probability of at least one success is

$$
1-(1-p)^k.
$$

## Task 7.1 — Predict the Effect of Restarts

Before coding, answer:

1. Why can random restart help with local maxima?
2. Does random restart change the behavior of one hill-climbing run?
3. What does it change instead?
4. If the probability of reaching the global optimum from a random start is $0.25$, what is the probability of at least one success after 4 independent runs?

Show your calculation.

**Your answer:**

In [ ]:
def random_restart_hill_climb(restarts: int, rng=None):
    if rng is None:
        rng = random.Random()

    best_state = None
    best_value = float("-inf")
    all_runs = []

    for run in range(restarts):
        start = rng.randint(DOMAIN_MIN, DOMAIN_MAX)
        final_state, final_value, trace = hill_climb(start)

        all_runs.append({
            "run": run + 1,
            "start": start,
            "final_state": final_state,
            "final_value": final_value,
            "moves": len(trace) - 1,
        })

        if final_value > best_value:
            best_state = final_state
            best_value = final_value

    return best_state, best_value, all_runs

## Task 7.2 — Small Random-Restart Experiment

Run 10 restarts with a fixed seed so your result is reproducible.

In [ ]:
rng = random.Random(42)

best_state, best_value, runs = random_restart_hill_climb(
    restarts=10,
    rng=rng
)

for row in runs:
    print(
        f"Run {row['run']:>2}: "
        f"start={row['start']:>3}, "
        f"final={row['final_state']:>3}, "
        f"value={row['final_value']:>8.3f}, "
        f"moves={row['moves']}"
    )

print("\nBest state:", best_state)
print("Best value:", round(best_value, 4))

## Task 7.3 — Interpret the 10-Restart Experiment

Answer:

1. How many different final peaks appeared?
2. Was the global optimum found?
3. Which run found the best result?
4. Why should you not judge the method using only one random experiment?
5. Why is fixing the random seed useful when comparing implementations?

**Your answers:**

# Part VIII — Estimate Success Rate Empirically

We now run many random starts to estimate how often ordinary hill climbing reaches the global optimum.

A run counts as successful when its final state has the same objective value as the known global optimum, allowing for tiny floating-point differences.

In [ ]:
def reaches_global_optimum(final_value, tolerance=1e-9):
    return abs(final_value - best_global_value) <= tolerance


def estimate_success_rate(trials=200, seed=7):
    rng = random.Random(seed)
    successes = 0
    values = []

    for _ in range(trials):
        start = rng.randint(DOMAIN_MIN, DOMAIN_MAX)
        _, final_value, _ = hill_climb(start)
        values.append(final_value)

        if reaches_global_optimum(final_value):
            successes += 1

    return {
        "trials": trials,
        "successes": successes,
        "success_rate": successes / trials,
        "mean_final_value": statistics.mean(values),
        "best_final_value": max(values),
        "worst_final_value": min(values),
    }

stats = estimate_success_rate()

for key, value in stats.items():
    print(f"{key}: {value}")

## Task 8.1 — Analyze Empirical Performance

Record:

| Measure | Result |
|---|---:|
| Number of trials |  |
| Global-optimum successes |  |
| Estimated success rate |  |
| Mean final objective |  |
| Best final objective |  |
| Worst final objective |  |

Then answer:

1. What does the success rate tell you that a single run does not?
2. Why is the mean final objective also useful?
3. Could two algorithms have the same success rate but different mean final objective values? Explain.
4. Why are repeated trials important when randomness is involved?

**Your answers:**

# Part IX — Compare Restart Budgets

We now compare different numbers of restarts.

For each restart budget, run several independent experiments and record how often the global optimum is found.

In [ ]:
def restart_success_experiment(restart_budget, repetitions=100, seed=123):
    master_rng = random.Random(seed)
    successes = 0

    for _ in range(repetitions):
        experiment_seed = master_rng.randint(0, 10**9)
        rng = random.Random(experiment_seed)

        _, best_value, _ = random_restart_hill_climb(
            restart_budget,
            rng
        )

        if reaches_global_optimum(best_value):
            successes += 1

    return successes / repetitions


budgets = [1, 2, 5, 10, 20]

print("restarts | success rate")
for k in budgets:
    rate = restart_success_experiment(k)
    print(f"{k:>8} | {rate:.3f}")

## Task 9.1 — Interpret Restart Budget

1. How did success rate change as the restart budget increased?
2. Why does more restarting usually improve robustness?
3. What is the computational trade-off?
4. Does a larger restart budget guarantee the global optimum in every finite experiment?
5. How would you choose a practical restart budget?

**Your answers:**

# Part X — Debugging Local Search

## Task 10.1 — Faulty Move Direction

A student writes:

```python
best_neighbor = min(neighbors(current), key=objective)
```

for a maximization problem.

1. What is wrong?
2. What behavior would you expect?
3. What should be used instead?

**Your answer:**

## Task 10.2 — Faulty Stopping Condition

A student writes:

```python
if objective(best_neighbor) >= objective(current):
    current = best_neighbor
```

with no protection against cycles.

1. What new type of move is allowed by `>=`?
2. Why can this be useful on a plateau?
3. Why can it also cause an infinite loop?
4. Give one simple strategy for limiting this risk.

**Your answer:**

## Task 10.3 — Faulty Random Restart

A student writes:

```python
start = 50
for _ in range(20):
    hill_climb(start)
```

and calls this random-restart hill climbing.

1. Why is this not actually random restart?
2. What must change between runs?
3. Why would all 20 runs otherwise produce the same result in our deterministic implementation?

**Your answer:**

# Part XI — Personalized Variation

Use the last digit of your student ID.

- `0–3`: use neighborhood radius 1
- `4–6`: use neighborhood radius 2
- `7–9`: use neighborhood radius 3

A radius-$r$ neighborhood is

$$
N_r(x)=\{x-r,\ldots,x-1,x+1,\ldots,x+r\}
$$

restricted to the valid domain.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

if LAST_DIGIT is not None:
    if 0 <= LAST_DIGIT <= 3:
        RADIUS = 1
    elif 4 <= LAST_DIGIT <= 6:
        RADIUS = 2
    elif 7 <= LAST_DIGIT <= 9:
        RADIUS = 3
    else:
        raise ValueError("LAST_DIGIT must be between 0 and 9")

    print("Assigned neighborhood radius:", RADIUS)

## Task 11.1 — Implement the Assigned Neighborhood

Complete the function.

In [ ]:
def radius_neighbors(x, radius):
    # TODO:
    # return every valid integer state within the given radius,
    # excluding x itself
    raise NotImplementedError

## Task 11.2 — Predict Before Running

Before modifying hill climbing to use your assigned radius, write:

- **Assigned radius:**  
- **Do I expect larger neighborhoods to reduce some local-maxima problems?**  
- **Could larger neighborhoods increase computation per step? Why?**  
- **Predicted effect on number of moves:**  

Then implement and run your variant.

In [ ]:
def hill_climb_radius(start, radius):
    current = start
    trace = [(current, objective(current))]

    while True:
        candidates = radius_neighbors(current, radius)
        best_neighbor = max(candidates, key=objective)

        if objective(best_neighbor) <= objective(current):
            return current, objective(current), trace

        current = best_neighbor
        trace.append((current, objective(current)))


if LAST_DIGIT is not None:
    personal_results = []
    for s in [5, 20, 35, 55, 75, 95]:
        end_x, end_value, trace = hill_climb_radius(s, RADIUS)
        personal_results.append((s, end_x, end_value, len(trace) - 1))

    print("start | final | value | moves")
    for row in personal_results:
        print(
            f"{row[0]:>5} | {row[1]:>5} | "
            f"{row[2]:>7.3f} | {row[3]:>5}"
        )

## Task 11.3 — Explain the Personalized Result

1. How did your assigned neighborhood radius affect the final states?
2. Did it help escape any peak that radius 1 could not escape?
3. Did it reduce or increase the number of moves?
4. Why can changing the neighborhood alter search behavior without changing the objective function?
5. Which part of a local-search problem specification did you modify: state space, objective, neighborhood, or all three?

**Your answers:**

# Part XII — Individual Understanding Check

Your instructor may ask one short question about your notebook.

Possible prompts:

- Show me where your hill-climbing code chooses the best neighbor.
- Why does the algorithm stop at a local maximum?
- Why can two starting states reach different answers?
- What is the difference between a local and a global optimum?
- Why can random restart improve success rate?
- What does the random seed control?
- Why are repeated trials important?
- What changed when you increased the neighborhood radius?

> You are expected to explain the **AI concept represented by the code**, not memorize Python syntax.

# Reflection

Answer concisely but precisely.

### R1 — Path vs. State
Why is hill climbing usually more concerned with the quality of the final state than with the path used to reach it?

**Answer:**

### R2 — Completeness
Is ordinary hill climbing complete for general optimization problems? Explain.

**Answer:**

### R3 — Optimality
Is ordinary hill climbing guaranteed to reach the global optimum? Explain.

**Answer:**

### R4 — Random Restart
Why can random restart make hill climbing more reliable without changing the basic local move rule?

**Answer:**

### R5 — Search Performance
Which metrics would you report when comparing two stochastic local-search methods?

**Answer:**

# Submission Checklist

Before submitting, verify that your notebook contains:

- [ ] state/objective/neighborhood explanations;
- [ ] manual hill-climbing prediction;
- [ ] working hill-climbing implementation;
- [ ] local-vs-global comparison;
- [ ] multi-start experiment;
- [ ] explanation of local maximum, plateau, and ridge;
- [ ] random-restart experiment;
- [ ] empirical success-rate analysis;
- [ ] restart-budget comparison;
- [ ] debugging answers;
- [ ] personalized neighborhood-radius experiment;
- [ ] prediction before the personalized run;
- [ ] reflection answers;
- [ ] visible outputs from important code cells.

Suggested filename:

```text
Lab05_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | Hill climbing, random restart, and personalized variant work correctly |
| **Algorithmic justification** | **3.0** | Explains objective, neighborhood, stopping condition, local optima, plateaus, and restart logic |
| **Experimental analysis** | **2.0** | Interprets multi-start, success-rate, and restart-budget results |
| **Trace / prediction / debugging** | **1.0** | Manual trace, predictions, and diagnosis of faulty local-search logic |
| **Individual understanding check** | **1.0** | Short explanation of selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** Correct code without adequate explanation earns only a limited portion of the marks.

## Key Takeaways

- Hill climbing evaluates the quality of **neighboring states** rather than building a full search tree.
- It moves toward improvement until no strictly better neighbor exists.
- It can stop at a **local maximum** that is not globally optimal.
- Plateaus and ridges can also make local search difficult.
- The result can depend strongly on the initial state.
- Random restart improves robustness by exploring multiple regions of the landscape.
- Local-search evaluation should use repeated trials and report measures such as:
  - success rate;
  - mean final objective;
  - best/worst final objective;
  - number of moves;
  - computational effort.

The next lab will introduce **Simulated Annealing**, which can sometimes accept worse moves to escape local optima.